手写LayerNorm

In [ ]:
"""
输入：
x: [B, T, C]
"""

In [ ]:
import torch
import torch.nn as nn

class MyLayerNorm(nn.Module):
    def __init__(self, hidden_size, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(hidden_size))
        self.beta = nn.Parameter(torch.zeros(hidden_size))
    
    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        x_norm = (x - mean) / torch.sqrt(var + self.eps)
        return x_norm * self.gamma + self.beta

手写RMSNorm

In [ ]:
import torch
import torch.nn as nn

class MyRMSNorm(nn.Module):
    def __init__(self, hidden_size, eps=1e-5):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(hidden_size))
        self.eps = eps
    
    def forward(self, x):
        rms = (x ** 2).mean(dim=-1, keepdim=True)
        x_norm = x / torch.sqrt(rms + self.eps)
        return x_norm * self.gamma

手写DPO loss

In [ ]:
import torch
import torch.nn.functional as F

def dpo_loss(
    policy_chosen_logps,
    policy_rejected_logps,
    ref_chosen_logps,
    ref_rejected_logps,
    beta=0.1
):
    # 1. policy model 对 chosen/rejected 的偏好强度
    policy_logratios = policy_chosen_logps - policy_rejected_logps

    # 2. reference model 对 chosen/rejected 的偏好强度
    ref_logratios = ref_chosen_logps - ref_rejected_logps

    # 3. policy 相对 reference 的偏好提升
    logits = beta * (policy_logratios - policy_rejected_logps)

    # 4. DPO loss
    loss = -F.logsigmoid(logits).mean()

    return loss
# logits要乘beta，loss要取mean()